In [ ]:
import os

INPUT_BASE = '/kaggle/input/datasets/msebastiaanh/7gcn-yeet/HAABSA7GCN'

assert os.path.isdir(f'{INPUT_BASE}/src'), f"src/ not found under {INPUT_BASE}"
assert os.path.isdir(f'{INPUT_BASE}/data'), f"data/ not found under {INPUT_BASE}"
print('src/:', sorted(os.listdir(f'{INPUT_BASE}/src')))
print('data/:', sorted(os.listdir(f'{INPUT_BASE}/data')))

In [ ]:
import shutil, os, sys

WORK = '/kaggle/working'
os.makedirs(f'{WORK}/data', exist_ok=True)

# Copy all data files (.const, .dep, ontology, rem*.csv, cross_features_*.pt)
for f in os.listdir(f'{INPUT_BASE}/data'):
    shutil.copy(f'{INPUT_BASE}/data/{f}', f'{WORK}/data/{f}')

shutil.copy(f'{INPUT_BASE}/cooc_matrix_final2.csv', f'{WORK}/cooc_matrix_final2.csv')
shutil.copy(f'{INPUT_BASE}/src/test_ontology_keys.csv', f'{WORK}/test_ontology_keys.csv')

for m in ['cross_example.py', 'fusion.py']:
    shutil.copy(f'{INPUT_BASE}/src/{m}', f'{WORK}/{m}')
sys.path.insert(0, WORK)

os.chdir(WORK)
print('cwd:', os.getcwd())
print('working/ contents:', sorted(os.listdir(WORK)))

# verify const files
for year in ['2015', '2016']:
    for split in ['train', 'test']:
        p = f'data/{split}{year}restaurant.txt.const'
        assert os.path.exists(p), f'MISSING: {p}'

for year in ['2015', '2016']:
    assert os.path.exists(f'data/cross_features_{year}.pt'), f'cross_features_{year}.pt missing in working'
assert os.path.exists('cross_example.py') and os.path.exists('fusion.py'), 'module files missing in working'
print('const + cooc + ontology + cross_features + modules OK')

In [ ]:
!pip install pytorch_transformers==1.2.0 --quiet

In [ ]:
import torch, sys
print('python:', sys.version.split()[0])
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
import pytorch_transformers
print('pytorch_transformers:', pytorch_transformers.__version__)

In [ ]:
import nbformat
NB_PATH = f'{INPUT_BASE}/src/7GCN.ipynb'
nb = nbformat.read(NB_PATH, as_version=4)

DEFINITION_CELLS = list(range(0, 25))
for i, cell in enumerate(nb.cells):
    if i not in DEFINITION_CELLS or cell.cell_type != 'code':
        continue
    src = ''.join(cell.source)
    if not src.strip():
        continue
    try:
        exec(src, globals())
        print(f"✓ cell {i:2d} OK ({len(src)} chars)")
    except Exception as e:
        print(f"✗ cell {i:2d} FAILED: {type(e).__name__}: {e}")
        raise

In [ ]:
import pandas as pd
print('Loading co-occurrence matrix...')
cooc = pd.read_csv('cooc_matrix_final2.csv', index_col=0)
print(f'  cooc shape: {cooc.shape}')

print('Loading ontology words...')
onto_words_df = pd.read_csv('test_ontology_keys.csv', sep=';')
onto_words = [item for sublist in onto_words_df.values.tolist() for item in sublist]
onto_words = list(dict.fromkeys(onto_words))
print(f'  ontology words count: {len(onto_words)}')

In [ ]:
import os, json, time, gc

EPOCHS     = 15
SEED       = 7
XSIM_TOP_K = 20
CONCAT_DROPOUT_FIXED = 0.2285714285714286
YEARS      = ['2015', '2016']
OUTDIR     = '/kaggle/working/ablation_7gcn_results'
os.makedirs(OUTDIR, exist_ok=True)

HP = {'lr': 1.0490678654396277e-05, 'dropout': 0.20,
      'l2reg': 0.0014773338109036142, 'fusion_dropout': 0.30}

# Each row is ONE complete run-config. Comment out rows to skip them per commit.
# columns:            name                const  xcat   xsim   fusion
ABLATION = [
    #{'name': '4gcn',       'constgcn': False, 'xcatgcn': False, 'xsimgcn': False, 'fusion': 'gated' },
    #{'name': '5gcn_const', 'constgcn': True,  'xcatgcn': False, 'xsimgcn': False, 'fusion': 'gated' },
    #{'name': '5gcn_const', 'constgcn': True,  'xcatgcn': False, 'xsimgcn': False, 'fusion': 'concat'},
    #{'name': '5gcn_xcat',  'constgcn': False, 'xcatgcn': True,  'xsimgcn': False, 'fusion': 'gated' },
    #{'name': '5gcn_xcat',  'constgcn': False, 'xcatgcn': True,  'xsimgcn': False, 'fusion': 'concat'},
    #{'name': '5gcn_xsim',  'constgcn': False, 'xcatgcn': False, 'xsimgcn': True,  'fusion': 'gated' },
    #{'name': '5gcn_xsim',  'constgcn': False, 'xcatgcn': False, 'xsimgcn': True,  'fusion': 'concat'},
    #{'name': '6gcn_cx',    'constgcn': True,  'xcatgcn': True,  'xsimgcn': False, 'fusion': 'gated' },
    #{'name': '6gcn_cx',    'constgcn': True,  'xcatgcn': True,  'xsimgcn': False, 'fusion': 'concat'},
    #{'name': '6gcn_cs',    'constgcn': True,  'xcatgcn': False, 'xsimgcn': True,  'fusion': 'gated' },
    #{'name': '6gcn_cs',    'constgcn': True,  'xcatgcn': False, 'xsimgcn': True,  'fusion': 'concat'},
    #{'name': '6gcn_xs',    'constgcn': False, 'xcatgcn': True,  'xsimgcn': True,  'fusion': 'gated' },
    #{'name': '6gcn_xs',    'constgcn': False, 'xcatgcn': True,  'xsimgcn': True,  'fusion': 'concat'},
    #{'name': '7gcn',       'constgcn': True,  'xcatgcn': True,  'xsimgcn': True,  'fusion': 'concat'},
]

def combo_str(a):
    on = [m for m,k in [('Const','constgcn'),('XCat','xcatgcn'),('XSim','xsimgcn')] if a[k]]
    mods = '+'.join(['core4'] + on)
    return (f'{a["name"]} | modules: {mods} '
            f'(const={a["constgcn"]}, xcat={a["xcatgcn"]}, xsim={a["xsimgcn"]}) '
            f'| fusion={a["fusion"]} | k={XSIM_TOP_K}')

for arch in ABLATION:
    for YEAR in YEARS:
        fusion = arch['fusion']
        tag = f'{arch["name"]}_{fusion}_{YEAR}_seed{SEED}'
        out_json = f'{OUTDIR}/{tag}.json'
        if os.path.exists(out_json) and EPOCHS == 15:
            prev = json.load(open(out_json))
            print(f'[skip] {tag}: val={prev["max_val_acc"]:.4f} '
                  f'test={prev["test_acc"]:.4f} f1={prev["test_f1"]:.4f}')
            continue

        print(f'\n{"="*78}\n>>> RUNNING: {combo_str(arch)}  year={YEAR}\n{"="*78}')

        global opt
        opt = get_args(
            model_type='tri_gcn',
            tgcn=True, semgcn=True, lexgcn=True, knogcn=True,
            constgcn=arch['constgcn'], xcatgcn=arch['xcatgcn'], xsimgcn=arch['xsimgcn'],
            fusion_type=fusion,
            fusion_dropout=float(HP['fusion_dropout']),
            xsim_top_k=XSIM_TOP_K,
            year=YEAR, seed=SEED,
            learning_rate=float(HP['lr']), dropout=float(HP['dropout']),
            concat_dropout=CONCAT_DROPOUT_FIXED, l2reg=float(HP['l2reg']),
            num_epoch=EPOCHS, batch_size=4,
            save_models='none', use_ensemble=True,
            cooc=cooc, onto_words=onto_words,
            do_train=True, do_eval=True,
            path=f'{OUTDIR}/tmp_{tag}',
        )
        opt.device = torch.device('cuda')

        assert opt.use_ensemble is True
        assert opt.fusion_type == fusion
        assert opt.xsim_top_k == 20
        assert opt.seed == 7
        assert (opt.modules['constgcn'], opt.modules['xcatgcn'], opt.modules['xsimgcn']) \
               == (arch['constgcn'], arch['xcatgcn'], arch['xsimgcn']), \
               'module flags did not propagate to opt'

        t0 = time.time()
        max_val_acc, test_acc, test_f1 = main(opt)
        elapsed = time.time() - t0

        result = {
            'arch': arch['name'],
            'modules': {k: arch[k] for k in ('constgcn','xcatgcn','xsimgcn')},
            'fusion_type': fusion,
            'max_val_acc': float(max_val_acc), 'test_acc': float(test_acc),
            'test_f1': float(test_f1), 'elapsed_sec': round(elapsed,1),
            'seed': SEED, 'year': YEAR, 'epochs': EPOCHS,
            'learning_rate': HP['lr'], 'dropout': HP['dropout'],
            'concat_dropout': CONCAT_DROPOUT_FIXED, 'l2reg': HP['l2reg'],
            'fusion_dropout': HP['fusion_dropout'], 'xsim_top_k': XSIM_TOP_K,
        }
        if EPOCHS == 15:
            with open(out_json, 'w') as f:
                json.dump(result, f, indent=2)

        print(f'[done] {combo_str(arch)}  year={YEAR}\n'
              f'       -> val={max_val_acc:.4f}  test={test_acc:.4f}  '
              f'f1={test_f1:.4f}  ({elapsed/60:.1f} min)')
        gc.collect(); torch.cuda.empty_cache()

# summary grid
print(f'\n{"="*78}\nABLATION SUMMARY (seed {SEED}, {EPOCHS} ep, k={XSIM_TOP_K})\n{"="*78}')
for arch in ABLATION:
    for YEAR in YEARS:
        p = f'{OUTDIR}/{arch["name"]}_{arch["fusion"]}_{YEAR}_seed{SEED}.json'
        if os.path.exists(p):
            r = json.load(open(p))
            print(f'  {arch["name"]:11s} {arch["fusion"]:6s} {YEAR}: '
                  f'val={r["max_val_acc"]:.4f} test={r["test_acc"]:.4f} f1={r["test_f1"]:.4f}')